# MUSIC3D 3D Visualization Demo

This notebook demonstrates 3D visualization of MUSIC3D HDF5 data including:
- Sinogram data visualization
- 3D Reconstruction volume rendering
- Segmentation overlays
- Interactive 3D exploration tools

## Setup and Dependencies

In [13]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print('Plotly not available - install with: pip install plotly')

try:
    from scipy import ndimage
    from skimage import measure
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False
    print('scikit-image not available - install with: pip install scikit-image')

## Load the utilities module

In [14]:
from music3d_utilities import MUSIC3DLoader, MUSIC3DVisualizer

## Configuration and Data Loading

In [15]:
DATA_ROOT = 'E:/Data/MUSIC3D_HDF5'
SAMPLE_NAME = 'Sample_31102016'
SPECTRUM_TYPE = 'fullSpectrum'

loader = MUSIC3DLoader(DATA_ROOT)
available_samples = loader.list_samples()
print(f'Available samples: {available_samples}')
print(f'Using sample: {SAMPLE_NAME}')

Available samples: ['MUSIC3D_HDF5']
Using sample: Sample_31102016


## Load Reconstruction Data

In [16]:
recon_data, recon_shape = loader.load_reconstruction(SAMPLE_NAME, SPECTRUM_TYPE)

if recon_data is not None:
    print(f'Reconstruction shape: {recon_shape}')
    print(f'Reconstruction dtype: {recon_data.dtype}')
    print(f'Value range: [{recon_data.min():.3f}, {recon_data.max():.3f}]')
else:
    print('Failed to load reconstruction')

Reconstruction not found at E:\Data\MUSIC3D_HDF5\Sample_31102016\fullSpectrum\reconstruction\reconstruction.h5
Failed to load reconstruction


## Visualization 1: Orthogonal Slices

In [17]:
if recon_data is not None:
    MUSIC3DVisualizer.create_volume_slices_matplotlib(
        recon_data,
        title=f'Reconstruction Orthogonal Slices ({SAMPLE_NAME})'
    )

## Visualization 2: Maximum Intensity Projection (MIP)

In [18]:
if recon_data is not None:
    MUSIC3DVisualizer.visualize_mip(
        recon_data,
        title=f'Maximum Intensity Projections ({SAMPLE_NAME})'
    )

## Visualization 3: Interactive 3D Isosurface (Plotly)

In [19]:
if recon_data is not None:
    data_to_viz = recon_data
    if data_to_viz.shape[0] > 256:
        step = max(1, data_to_viz.shape[0] // 128)
        data_to_viz = data_to_viz[::step, ::step, ::step]
        print(f'Downsampled to {data_to_viz.shape} for visualization')
    MUSIC3DVisualizer.create_isosurface_plotly(
        data_to_viz,
        threshold=0.4,
        title=f'3D Reconstruction Surface ({SAMPLE_NAME})'
    )

## Load and Visualize Segmentation

In [20]:
seg_data, seg_shape = loader.load_segmentation(SAMPLE_NAME, 'manualSegmentation')

if seg_data is not None:
    print(f'Segmentation shape: {seg_shape}')
    print(f'Number of unique labels: {len(np.unique(seg_data))}')
else:
    print('No manual segmentation found')

No manual segmentation found


## Visualization 4: Segmentation Overlay

In [21]:
if recon_data is not None and seg_data is not None:
    MUSIC3DVisualizer.visualize_segmentation_overlay(
        recon_data,
        seg_data,
        alpha=0.4,
        title=f'Reconstruction with Segmentation ({SAMPLE_NAME})'
    )

## Load and Visualize Sinogram

In [22]:
sino_data, sino_shape = loader.load_sinogram(SAMPLE_NAME, SPECTRUM_TYPE)

if sino_data is not None:
    print(f'Sinogram shape: {sino_shape}')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(sino_data[sino_data.shape[0]//2, :, :], cmap='gray')
    axes[0].set_title('Sinogram Slice (middle)')
    axes[0].axis('off')
    axes[1].imshow(sino_data[:, :, sino_data.shape[2]//2], cmap='gray', aspect='auto')
    axes[1].set_title('Sinogram Slice (detector)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Failed to load sinogram')

Sinogram not found at E:\Data\MUSIC3D_HDF5\Sample_31102016\fullSpectrum\projections\sinogram.h5
Failed to load sinogram


## Summary

This notebook demonstrates:
- Loading MUSIC3D HDF5 data
- Orthogonal slice visualization
- Maximum Intensity Projections (MIP)
- Interactive 3D isosurface visualization
- Segmentation overlays
- Sinogram/projection data visualization

For integration into your main software, use `music3d_utilities.py`.